# Compact Real-Data ProtoHedge Panel Tests - PyTorch

This is a smaller exploratory counterpart to `protohedge-paper-final.ipynb`. It uses only the predeclared `K in {10,25}` spot-delta prototype grid, but otherwise enforces the corrected scientific protocol: fixed-contract episodes, chronological pre-window splits, premium-excluding liability-offset terminology, validation-only model selection, one-time test evaluation, and paired circular-block uncertainty. Use the full final notebook for paper results.


## How This Maps To The Paper

The current single-panel paper draft already supports the main claim that ProtoHedge can preserve most of vanilla Deep Hedging's performance while giving a more interpretable decision structure. This notebook is the next external-validity step: apply the same evaluation logic to a broader panel of ETFs and single-name underlyings and summarize whether the tradeoff survives across markets.


## 0. Setup

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

cwd = Path.cwd().resolve()
REPO_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
REPO_PARENT = REPO_ROOT.parent
assert (REPO_ROOT / 'world_real_torch.py').exists(), f'Could not find local repo module at {REPO_ROOT / "world_real_torch.py"}'

repo_parent_str = str(REPO_PARENT)
sys.path = [p for p in sys.path if Path(p or '.').resolve() != REPO_PARENT]
sys.path.insert(0, repo_parent_str)

for mod_name in list(sys.modules):
    if mod_name == 'deephedging' or mod_name.startswith('deephedging.'):
        del sys.modules[mod_name]

from deephedging.real_data_sweep_torch import run_real_data_sweep, MODEL_SELECTION_VERSION
from deephedging.outcome_metrics import OUTCOME_DEFINITION_VERSION
from deephedging.real_data_analysis_torch import (
    evaluate_saved_artifact,
    evaluate_saved_baselines,
    build_regime_frame,
    summarize_by_regime,
    prototype_usage_table,
    prototype_usage_by_regime,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)


## 1. Panel Configuration

In [ ]:
PANEL_DIR = REPO_ROOT / 'Data' / 'NEW_PANEL'
MANIFEST_PATH = PANEL_DIR / 'panel_manifest.csv'
PANEL_OUTPUT_ROOT = REPO_ROOT / '.deephedging_real_runs' / 'new_data_panel_scientific_v2'
PANEL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PANEL_SUMMARY_DIR = PANEL_OUTPUT_ROOT / 'panel_summary'
PANEL_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

PANEL_TICKERS = None
RUN_PANEL_SWEEP = True
SEEDS = (1234, 2345, 3456)
EPOCHS = 800
NORMALIZE_REAL_WORLD = True
HEDGE_MODE = 'step'
USE_POSITION_BOUNDS = True

TRADE_BOUNDS = {'lbnd_as': -1.0, 'ubnd_as': 1.0, 'lbnd_av': -1.0, 'ubnd_av': 1.0}
POSITION_BOUNDS = {'lbnd_delta_s': -1.0, 'ubnd_delta_s': 1.0, 'lbnd_delta_v': -1.0, 'ubnd_delta_v': 1.0}
TRAIN_SELECTION_CFG = {
    'selection_metric': 'val_loss',
    'selection_alpha_action_abs': 0.005,
    'selection_alpha_delta_abs': 0.010,
    'selection_alpha_bound_occupancy': 0.100,
    'selection_alpha_path_bound_touch': 0.0,
}
TRAIN_REG_CFG = {'action_penalty_weight': 0.001, 'delta_penalty_weight': 0.002}
ROBUST_SCREEN_CFG = {'max_bound_occupancy': 0.50, 'max_path_touch_rate': None}
PROTOTYPE_COUNTS = (10, 25)
PROTOTYPE_SOURCES = ('spot_delta',)
WEIGHTED_SIMILARITY_OPTIONS = (False,)
LEARN_DISTANCE_FEATURE_WEIGHTS_OPTIONS = (False,)
BOOTSTRAP_REPETITIONS = 2000
BOOTSTRAP_BLOCK_LENGTH = 20

PANEL_BASELINE_LABELS = {
    'unhedged': 'Unhedged',
    'spot_delta': 'Spot-Delta',
    'spot_delta_band': 'Spot-Delta Band',
    'vanilla': 'Vanilla DH',
}
PANEL_PROTO_SELECTION_LABELS = {
    'best_screened_proto_mean': 'ProtoHedge (Validation-Mean)',
    'best_screened_proto_cvar05': 'ProtoHedge (Validation-Tail)',
}
PANEL_MODEL_ORDER = [*PANEL_BASELINE_LABELS.values(), *PANEL_PROTO_SELECTION_LABELS.values()]
DEEP_DIVE_TICKER = 'QQQ'

print('manifest:', MANIFEST_PATH)
print('corrected panel output root:', PANEL_OUTPUT_ROOT)
print('seeds:', SEEDS, '| epochs:', EPOCHS)


## 2. Load And Validate The Panel Manifest

In [ ]:
assert MANIFEST_PATH.exists(), f'Missing manifest: {MANIFEST_PATH}. Run the cleaning notebook first.'
manifest = pd.read_csv(MANIFEST_PATH)
assert manifest['split_version'].eq('chronological-prewindow-v1').all(), 'Panel must use fixed chronological splits.'
assert manifest['option_path_version'].eq('fixed-optionid-v1').all(), 'Panel must use contract-consistent option paths.'
manifest = manifest[manifest['status'] == 'ok'].copy().reset_index(drop=True)
assert not manifest.empty, 'No successfully processed tickers found in the panel manifest.'

if PANEL_TICKERS is not None:
    wanted = {str(t).upper() for t in PANEL_TICKERS}
    manifest = manifest[manifest['ticker'].isin(wanted)].copy().reset_index(drop=True)

for col in ['clean_spot_path', 'feature_path', 'episode_path', 'split_path', 'episode_metadata_path']:
    manifest[col] = manifest[col].apply(lambda x: str(Path(x).resolve()))
    missing = [p for p in manifest[col] if not Path(p).exists()]
    assert not missing, f'Missing files in manifest column {col}: {missing[:3]}'

manifest


In [ ]:
shape_rows = []
for _, row in manifest.iterrows():
    arr = np.load(row['episode_path'], mmap_mode='r')
    shape_rows.append({
        'ticker': row['ticker'],
        'shape': tuple(arr.shape),
        'dtype': str(arr.dtype),
        'spot0_mean': float(np.mean(arr[:, 0, 0])),
        'call_delta_mean': float(np.mean(arr[:, :, 2])),
        'ivol_mean': float(np.mean(arr[:, :, 4])),
    })
shape_df = pd.DataFrame(shape_rows)
display(shape_df)
assert all(s[1] == 20 and s[2] >= 5 for s in shape_df['shape'])


## 3. Panel Data Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
show = manifest.sort_values('ticker')
axes[0].bar(show['ticker'], show['n_episodes'])
axes[0].set_title('Episodes by ticker')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(show['ticker'], show['n_feature_rows'])
axes[1].set_title('Clean daily feature rows by ticker')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## 4. Run the Compact Validation-Selected Sweeps

Each candidate is trained and evaluated on training/validation periods only. The two predeclared validation selectors are then frozen, after which baselines, Deep Hedging, and selected ProtoHedge policies are evaluated once on test.


In [ ]:
def ticker_output_dir(ticker):
    return PANEL_OUTPUT_ROOT / f'{ticker}_compact_800_scientific_v2'


def run_one_ticker(row):
    ticker = row['ticker']
    out_dir = ticker_output_dir(ticker)
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'===== {ticker}: running compact validation-selected sweep =====')
    return run_real_data_sweep(
        data_path=row['episode_path'],
        split_path=row['split_path'],
        episode_metadata_path=row['episode_metadata_path'],
        output_dir=out_dir,
        samples=None,
        seeds=SEEDS,
        epochs=EPOCHS,
        prototype_counts=PROTOTYPE_COUNTS,
        prototype_sources=PROTOTYPE_SOURCES,
        weighted_similarity_options=WEIGHTED_SIMILARITY_OPTIONS,
        learn_distance_feature_weights_options=LEARN_DISTANCE_FEATURE_WEIGHTS_OPTIONS,
        risk_measures=('cvar',),
        normalize=NORMALIZE_REAL_WORLD,
        hedge_mode=HEDGE_MODE,
        position_bounds=USE_POSITION_BOUNDS,
        trade_bounds=TRADE_BOUNDS,
        cumulative_bounds=POSITION_BOUNDS,
        selection_metric=TRAIN_SELECTION_CFG['selection_metric'],
        selection_alpha_action_abs=TRAIN_SELECTION_CFG['selection_alpha_action_abs'],
        selection_alpha_delta_abs=TRAIN_SELECTION_CFG['selection_alpha_delta_abs'],
        selection_alpha_bound_occupancy=TRAIN_SELECTION_CFG['selection_alpha_bound_occupancy'],
        selection_alpha_path_bound_touch=TRAIN_SELECTION_CFG['selection_alpha_path_bound_touch'],
        action_penalty_weight=TRAIN_REG_CFG['action_penalty_weight'],
        delta_penalty_weight=TRAIN_REG_CFG['delta_penalty_weight'],
        max_bound_occupancy=ROBUST_SCREEN_CFG['max_bound_occupancy'],
        max_path_touch_rate=ROBUST_SCREEN_CFG['max_path_touch_rate'],
        tuned_baseline_metric='liability_offset_mean',
        bootstrap_repetitions=BOOTSTRAP_REPETITIONS,
        bootstrap_block_length=BOOTSTRAP_BLOCK_LENGTH,
    )


if RUN_PANEL_SWEEP:
    panel_metrics = {row['ticker']: run_one_ticker(row) for _, row in manifest.iterrows()}
else:
    print('RUN_PANEL_SWEEP=False, skipping sweep execution.')


## 5. Aggregate The Cross-Ticker Results

In [ ]:
panel_rows = []
for _, manifest_row in manifest.iterrows():
    ticker = manifest_row['ticker']
    out_dir = ticker_output_dir(ticker)
    comp = pd.read_csv(out_dir / 'paper_model_comparison.csv')
    best = pd.read_csv(out_dir / 'paper_best_models.csv')
    assert comp['outcome_definition'].eq(OUTCOME_DEFINITION_VERSION).all()
    assert best['selection_split'].eq('validation').all()
    assert best['model_selection_version'].eq(MODEL_SELECTION_VERSION).all()
    for model, label in PANEL_BASELINE_LABELS.items():
        row = comp[comp['model'] == model]
        if not row.empty:
            item = row.iloc[0].to_dict()
            item.update(ticker=ticker, paper_label=label)
            panel_rows.append(item)
    for selection, label in PANEL_PROTO_SELECTION_LABELS.items():
        row = best[best['selection'] == selection]
        assert len(row) == 1
        item = row.iloc[0].to_dict()
        item.update(ticker=ticker, paper_label=label)
        panel_rows.append(item)

panel_df = pd.DataFrame(panel_rows).sort_values(['ticker', 'paper_label']).reset_index(drop=True)
panel_df.to_csv(PANEL_SUMMARY_DIR / 'panel_model_metrics.csv', index=False)
display(panel_df[[
    'ticker', 'paper_label', 'liability_offset_mean_avg',
    'liability_offset_cvar05_avg', 'liability_offset_rmse_avg',
    'liability_offset_downside_deviation_avg',
    'pct_at_any_position_bound_avg', 'passes_validation_robust_screen',
]])


In [ ]:
vanilla = panel_df[panel_df['paper_label'] == 'Vanilla DH'][[
    'ticker', 'liability_offset_mean_avg', 'liability_offset_cvar05_avg',
    'liability_offset_rmse_avg', 'pct_at_any_position_bound_avg',
]].rename(columns={
    'liability_offset_mean_avg': 'vanilla_mean',
    'liability_offset_cvar05_avg': 'vanilla_cvar',
    'liability_offset_rmse_avg': 'vanilla_rmse',
    'pct_at_any_position_bound_avg': 'vanilla_bound',
})
proto_vs_vanilla = panel_df[panel_df['paper_label'].str.startswith('ProtoHedge')].merge(vanilla, on='ticker')
proto_vs_vanilla['mean_difference_vs_vanilla'] = proto_vs_vanilla['liability_offset_mean_avg'] - proto_vs_vanilla['vanilla_mean']
proto_vs_vanilla['cvar_difference_vs_vanilla'] = proto_vs_vanilla['liability_offset_cvar05_avg'] - proto_vs_vanilla['vanilla_cvar']
proto_vs_vanilla['rmse_improvement_vs_vanilla'] = proto_vs_vanilla['vanilla_rmse'] - proto_vs_vanilla['liability_offset_rmse_avg']
proto_vs_vanilla['bound_reduction_vs_vanilla'] = proto_vs_vanilla['vanilla_bound'] - proto_vs_vanilla['pct_at_any_position_bound_avg']
display(proto_vs_vanilla[[
    'ticker', 'paper_label', 'mean_difference_vs_vanilla',
    'cvar_difference_vs_vanilla', 'rmse_improvement_vs_vanilla',
    'bound_reduction_vs_vanilla',
]])


In [ ]:
def metric_heatmap(metric, title, fmt='.3f', cmap='viridis'):
    pivot = panel_df.pivot(index='ticker', columns='paper_label', values=metric)
    plt.figure(figsize=(11, max(4, 0.45 * len(pivot))))
    sns.heatmap(pivot, annot=True, fmt=fmt, cmap=cmap)
    plt.title(title)
    plt.tight_layout()
    plt.show()
    return pivot

metric_heatmap('liability_offset_mean_avg', 'Mean liability offset by ticker and model')
metric_heatmap('liability_offset_cvar05_avg', '5% CVaR of liability offset by ticker and model', cmap='magma')
metric_heatmap('liability_offset_rmse_avg', 'Liability-offset RMSE by ticker and model', cmap='crest_r')
metric_heatmap('pct_at_any_position_bound_avg', 'Bound occupancy by ticker and model', fmt='.2f', cmap='rocket_r')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for ax, metric, title in [
    (axes[0, 0], 'mean_difference_vs_vanilla', 'Mean-offset difference'),
    (axes[0, 1], 'cvar_difference_vs_vanilla', 'CVaR difference'),
    (axes[1, 0], 'rmse_improvement_vs_vanilla', 'RMSE improvement'),
    (axes[1, 1], 'bound_reduction_vs_vanilla', 'Bound reduction'),
]:
    for label, grp in proto_vs_vanilla.groupby('paper_label'):
        ax.plot(grp['ticker'], grp[metric], marker='o', label=label)
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.set_title(title + ' vs Deep Hedging (positive favors ProtoHedge)')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)
axes[0, 0].legend()
plt.tight_layout()
plt.show()


The heatmaps above answer the broad paper question: does the ProtoHedge tradeoff generalize across underlyings? The first figure shows raw performance, the second highlights downside behavior, and the third shows whether a model leans heavily on hard position bounds. The line plots then recast the same information as explicit ProtoHedge-versus-vanilla gaps ticker by ticker.


## 6. Panel-Wide Paper Tables

In [ ]:
panel_main = panel_df[[
    'ticker', 'paper_label', 'liability_offset_mean_avg',
    'liability_offset_mean_std', 'liability_offset_cvar05_avg',
    'liability_offset_rmse_avg', 'liability_offset_downside_deviation_avg',
    'pct_at_any_position_bound_avg', 'passes_validation_robust_screen',
]].copy()
panel_main.to_csv(PANEL_SUMMARY_DIR / 'panel_main_table.csv', index=False)
display(panel_main)


## 7. Single-Ticker Deep Dive

In [ ]:
if DEEP_DIVE_TICKER not in set(manifest['ticker']):
    DEEP_DIVE_TICKER = manifest.iloc[0]['ticker']
deep_dir = ticker_output_dir(DEEP_DIVE_TICKER)
deep_metrics = pd.read_csv(deep_dir / 'sweep_metrics.csv')
deep_summary = pd.read_csv(deep_dir / 'sweep_test_summary.csv')
deep_best = pd.read_csv(deep_dir / 'paper_best_models.csv')
display(deep_summary[[
    'model', 'liability_offset_mean_avg', 'liability_offset_cvar05_avg',
    'liability_offset_rmse_avg', 'pct_at_any_position_bound_avg',
]])


In [ ]:
selected = deep_best[deep_best['selection'] == 'best_screened_proto_cvar05']
assert len(selected) == 1 and selected.iloc[0]['selection_split'] == 'validation'
TARGET_MODEL = selected.iloc[0]['model']
artifact_rows = deep_metrics[(deep_metrics['model'] == TARGET_MODEL) & (deep_metrics['split'] == 'test')]
assert not artifact_rows.empty
artifact_dir = Path(artifact_rows.iloc[0]['artifact_dir'])
bundle, proto_result, proto_metrics = evaluate_saved_artifact(artifact_dir, split='test')
baselines = evaluate_saved_baselines(artifact_dir, split='test')
vanilla_rows = deep_metrics[(deep_metrics['model'] == 'vanilla') & (deep_metrics['split'] == 'test')]
vanilla_bundle, vanilla_result, vanilla_metrics = evaluate_saved_artifact(Path(vanilla_rows.iloc[0]['artifact_dir']), split='test')
result_dict = {
    'proto_validation_tail': proto_result,
    'vanilla': vanilla_result,
    'spot_delta_band': baselines['spot_delta_band']['result'],
    'unhedged': baselines['unhedged']['result'],
}
regime_frame = build_regime_frame(bundle['eval_world'], result_dict)
regime_summary = pd.concat([
    summarize_by_regime(regime_frame, 'return_regime'),
    summarize_by_regime(regime_frame, 'vol_regime'),
    summarize_by_regime(regime_frame, 'drawdown_regime'),
], ignore_index=True)
regime_summary.to_csv(PANEL_SUMMARY_DIR / f'{DEEP_DIVE_TICKER}_regime_summary.csv', index=False)
regime_summary.head()


In [ ]:
for regime_type in ['return_regime', 'vol_regime', 'drawdown_regime']:
    sub = regime_summary[regime_summary['regime_type'] == regime_type]
    for metric in ['mean', 'cvar05', 'rmse', 'downside_deviation']:
        pivot = sub.pivot(index='regime', columns='series', values=metric)
        plt.figure(figsize=(8, 3))
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis')
        plt.title(f'{DEEP_DIVE_TICKER}: {metric} by {regime_type}')
        plt.tight_layout()
        plt.show()


In [ ]:
top_proto_df, full_proto_df = prototype_usage_table(bundle, proto_result, top_n=10)
usage_by_regime_df = prototype_usage_by_regime(proto_result, regime_frame)

top_proto_df.to_csv(PANEL_SUMMARY_DIR / f'{DEEP_DIVE_TICKER}_prototype_top10.csv', index=False)
usage_by_regime_df.to_csv(PANEL_SUMMARY_DIR / f'{DEEP_DIVE_TICKER}_prototype_usage_by_regime.csv', index=False)
print('saved prototype tables for', DEEP_DIVE_TICKER)
display(top_proto_df)


In [ ]:
plt.figure(figsize=(10, 4))
plot_df = top_proto_df.sort_values('usage_mean', ascending=False).head(10)
plt.bar(plot_df['prototype_index'].astype(str), plot_df['usage_mean'])
plt.title(f'{DEEP_DIVE_TICKER}: top prototype usage')
plt.xlabel('prototype index')
plt.ylabel('mean weight')
plt.tight_layout()
plt.show()

for regime_type in ['return_regime', 'vol_regime', 'drawdown_regime']:
    sub = usage_by_regime_df[usage_by_regime_df['regime_type'] == regime_type].copy()
    pivot = sub.pivot(index='regime', columns='prototype', values='mean_weight')
    plt.figure(figsize=(10, 3))
    sns.heatmap(pivot, annot=False, cmap='magma')
    plt.title(f'{DEEP_DIVE_TICKER}: prototype usage by {regime_type}')
    plt.tight_layout()
    plt.show()


This compact notebook is an exploratory robustness check. The submission tables and figures must come from `protohedge-paper-final.ipynb`, which runs the full grid and exports validation-selected test results with dependence-aware confidence intervals.
